In [3]:
import os
import re
import json
import pdfplumber
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [4]:
def clean_text(text):
    """Strips out hyphenated word splits, headers, footers, and page markers."""
    text = re.sub(r'(\w+)-\s*\n(\w+)', r'\1\2', text)
    text = re.sub(r'\s+', ' ', text)
    # Target and eliminate textbook metadata layout footprints
    text = re.sub(r'(Curiosity\s*—.*Grade\s*8|Science\s+\d+|Chapter\s+\d+|\bPage\s+\d+\b|Reprint\s+\d+-\d+)', '', text, flags=re.IGNORECASE)
    return text.strip()

In [5]:
def chunk_text(text, chunk_size=500, overlap=100):
    """Generates continuous text windows with a safe context cushion overlap."""
    chunks = []
    words = text.split()
    i = 0
    while i < len(words):
        chunk_words = words[i:i + chunk_size]
        chunk_str = " ".join(chunk_words)
        if len(chunk_str.strip()) > 40:
            chunks.append(chunk_str)
        i += (chunk_size - overlap)
    return chunks

In [11]:
def extract_and_index(pdf_path="science class 8.pdf", output_jsonl="class8_science.jsonl"):
    print("✨ Step 1: Extracting text from local PDF...")
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"Source file '{pdf_path}' not found in current directory directory.")
        
    documents = []
    with pdfplumber.open(pdf_path) as pdf:
        # Simple rule-based strategy mapping to identify layout splits across the uploaded document stream
        current_chapter = "Introductory Materials"
        chapter_counter = 0
        
        for idx, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text() or ""
            
            # Simple heuristic detection for chapter demarcations
            chapter_match = re.search(r'(Chapter\s+\d+)\s*—\s*([^\n\r]+)', page_text, re.IGNORECASE)
            if chapter_match:
                chapter_counter += 1
                current_chapter = f"Chapter {chapter_counter}: {chapter_match.group(2).strip()}"
                
            cleaned = clean_text(page_text)
            chunks = chunk_text(cleaned)
            
            for chunk_idx, chunk in enumerate(chunks):
                documents.append({
                    "id": f"doc_p{idx}_c{chunk_idx}",
                    "source_chapter": current_chapter,
                    "text": chunk
                })

In [ ]:
def build_corpus(pdf_folder, output_jsonl):
    documents = []
    # ... your PDF processing code goes here ...
    
    # Make sure this print matches the indentation level of your main function code (4 spaces)
    print(f" Saved {len(documents)} clean semantic text slices into '{output_jsonl}'")


    print("\n✨ Step 2: Generating Vector Embeddings & compiling FAISS database index...")
    embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    texts = [doc['text'] for doc in documents]
    embeddings = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    
    # Save assets to disk
    faiss.write_index(index, "science_index.faiss")
    with open("metadata.json", "w", encoding="utf-8") as f:
        json.dump(documents, f)
    print(" FAISS structural lookup tables successfully configured and committed to storage disk.")

if __name__ == "__main__":
    extract_and_index(r"C:\Users\ADITYA\Downloads\science class 8.pdf")

✨ Step 1: Extracting text from local PDF...


In [ ]:
import openai

In [14]:
class AITutorRAGPipeline:
    def __init__(self, index_path="science_index.faiss", metadata_path="metadata.json"):
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self.index = faiss.read_index(index_path)
        with open(metadata_path, "r", encoding="utf-8") as f:
            self.metadata = json.load(f)

In [15]:
def retrieve_context(self, query, k=3):
        query_vector = self.embedder.encode([query], convert_to_numpy=True)
        distances, indices = self.index.search(query_vector, k)
        return [self.metadata[idx] for idx in indices[0] if idx < len(self.metadata)]

In [20]:
def generate_response(self, query, api_key):
        openai.api_key = api_key
        contexts = self.retrieve_context(query, k=3)
        
        context_block = ""
        for c in contexts:
            context_block += f"[Source Track: {c['source_chapter']}]\nText Context: {c['text']}\n\n"
            system_instructions = (
            "You are an expert, supportive AI Tutor specialized strictly in NCERT Class 8 Science.\n"
            "Execution Constraints:\n"
            "1. Answer the query using ONLY the verified facts provided in the Context below.\n"
            "2. Keep explanations simple, clear, and perfectly tailored for an 8th-grade student.\n"
            "3. If the answer cannot be confidently found in the context, do not use outside knowledge. Reply EXACTLY with: "
            "'I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook.'\n"
            "4. Do not make any references to external data sources.\n"
        )
        
        try:
            response = openai.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": system_instructions},
                    {"role": "user", "content": f"Context:\n{context_block}\nQuestion: {query}\nAnswer:"}
                ],
                temperature=0.0
            )
            answer = response.choices[0].message.content
        except Exception as e:
            answer = f"Pipeline Processing Error: {str(e)}"
            
        return {
            "answer": answer, 
            "sources": list(set([c['source_chapter'] for c in contexts])) if "focused on Class 8" not in answer else []
        }

In [22]:
!pip install peft transformers datasets accelerate bitsandbytes


   ---------------------------------------- 0.0/680.7 kB ? eta -:--:--
   ---------------------------------------- 680.7/680.7 kB 5.5 MB/s  0:00:00
   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 555.1/555.1 kB 11.8 MB/s  0:00:00
   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   -- ------------------------------------- 3.4/55.4 MB 15.2 MB/s eta 0:00:04
   --- ------------------------------------ 4.5/55.4 MB 11.5 MB/s eta 0:00:05
   ---- ----------------------------------- 5.8/55.4 MB 9.2 MB/s eta 0:00:06
   ---- ----------------------------------- 6.8/55.4 MB 7.8 MB/s eta 0:00:07
   ----- ---------------------------------- 7.6/55.4 MB 7.4 MB/s eta 0:00:07
   ------ --------------------------------- 8.4/55.4 MB 6.5 MB/s eta 0:00:08
   ------ --------------------------------- 8.7/55.4 MB 6.0 MB/s eta 0:00:08
   ------ --------------------------------- 8.9/55.4 MB 5.6 MB/s eta 0:00:09
   ------ -----

In [3]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

C:\Users\ADITYA\anaconda3\envs\MachineLearning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
training_data = {
    "input_text": [
        "What are the two main crop seasons?",
        "Define pasteurisation.",
        "What is a polymer?",
        "Can you help me solve advanced multivariable calculus theorems?"
    ],
    "target_text": [
        "The two main crop seasons are Kharif crops and Rabi crops.",
        "The process of heating milk to about 70 degree Celsius for 15 to 30 seconds and then suddenly chilling it to prevent microbe growth.",
        "A polymer is a large unit formed by combining many small chemical units together.",
        "I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook."
    ]
}

def run_lora_alignment():
    print("🚀 Initializing Parameter-Efficient Fine-Tuning Sequence...")
    model_id = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
    
    # Define Dataset structural map layout
    dataset = Dataset.from_dict(training_data)
    
    def preprocess_function(examples):
        inputs = [f"tutor query: {text}" for text in examples["input_text"]]
        model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
        labels = tokenizer(text_target=examples["target_text"], max_length=128, truncation=True, padding="max_length")
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_dataset = dataset.map(preprocess_function, batched=True)
    
    # Configure PEFT/LoRA Target Matrix Properties 
    peft_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        inference_mode=False,
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=["q", "v"]
    )
    
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    
    # Corrected TrainingArguments parameters
    training_args = TrainingArguments(
        output_dir="./tutor_lora_model",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=5e-4,
        num_train_epochs=3,
        logging_steps=1,
        eval_strategy="no",  # ◄── FIXED: Changed from evaluation_strategy to eval_strategy
        save_strategy="no",
        fp16=False
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
    )
    
    print("Executing fine-tuning loops...")
    trainer.train()
    print("✨ LoRA parameters successfully trained and updated.")

if __name__ == "__main__":
    run_lora_alignment()

🚀 Initializing Parameter-Efficient Fine-Tuning Sequence...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2283.96it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Map: 100%|██████████| 4/4 [00:00<00:00, 69.21 examples/s]


trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451
Executing fine-tuning loops...


C:\Users\ADITYA\anaconda3\envs\MachineLearning\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,73.191177
2,72.019073
3,70.933891


✨ LoRA parameters successfully trained and updated.


In [18]:
%%writefile tutor_backend.py
import json
import faiss
import openai
from sentence_transformers import SentenceTransformer

Overwriting tutor_backend.py


In [19]:
class AITutorRAGPipeline:
    def __init__(self, index_path="science_index.faiss", metadata_path="metadata.json"):
        self.embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        self.index = faiss.read_index(index_path)
        with open(metadata_path, "r", encoding="utf-8") as f:
            self.metadata = json.load(f)
            
    def retrieve_context(self, query, k=3):
        query_vector = self.embedder.encode([query], convert_to_numpy=True)
        distances, indices = self.index.search(query_vector, k)
        return [self.metadata[idx] for idx in indices[0] if idx < len(self.metadata)]

    def generate_response(self, query, api_key):
        """This function name must match your evaluation script exactly!"""
        openai.api_key = api_key
        contexts = self.retrieve_context(query, k=3)
        
        context_block = ""
        for c in contexts:
            context_block += f"[Source Track: {c['source_chapter']}]\nText Context: {c['text']}\n\n"
            
        system_instructions = (
            "You are an expert, supportive AI Tutor specialized strictly in NCERT Class 8 Science.\n"
            "Execution Constraints:\n"
            "1. Answer the query using ONLY the verified facts provided in the Context below.\n"
            "2. Keep explanations simple, clear, and perfectly tailored for an 8th-grade student.\n"
            "3. If the answer cannot be confidently found in the context, do not use outside knowledge. Reply EXACTLY with: \n"
            "'I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook.'\n"
            "4. Do not make any references to external data sources.\n"
        )
        
        try:
            response = openai.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": system_instructions},
                    {"role": "user", "content": f"Context:\n{context_block}\nQuestion: {query}\nAnswer:"}
                ],
                temperature=0.0
            )
            answer = response.choices[0].message.content
        except Exception as e:
            answer = f"Pipeline Processing Error: {str(e)}"
            
        return {
            "answer": answer, 
            "sources": list(set([c['source_chapter'] for c in contexts])) if "focused on Class 8" not in answer else []
        }

In [22]:
import csv
import sys
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer


In [23]:
nltk.download('punkt', quiet=True)
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth_func = SmoothingFunction().method1

In [24]:
eval_dataset = [
    {"query": "What are the two main crop seasons?", "ground_truth": "The two main crop seasons are Kharif crops and Rabi crops."},
    {"query": "Define pasteurisation.", "ground_truth": "The process of heating milk to about 70 degree Celsius for 15 to 30 seconds and then suddenly chilling it to prevent microbe growth."},
    {"query": "What is a polymer?", "ground_truth": "A polymer is a large unit formed by combining many small chemical units together."},
    {"query": "Name a liquid metal.", "ground_truth": "Mercury is a liquid metal."},
    {"query": "What is the process of loosening and turning the soil called?", "ground_truth": "The process of loosening and turning the soil is called tilling or ploughing."},
    {"query": "What are pathogens?", "ground_truth": "Disease-causing microorganisms are called pathogens."},
    {"query": "What is calorific value?", "ground_truth": "The amount of heat energy produced on complete combustion of 1 kg of a fuel is called its calorific value."},
    {"query": "What is an ecosystem?", "ground_truth": "An ecosystem is made of all the plants, animals and microorganisms in an area along with non-living components like climate, soil and rivers."},
    {"query": "What is weeding?", "ground_truth": "The removal of unwanted plants from a field is called weeding."},
    {"query": "Define combustion.", "ground_truth": "A chemical process in which a substance reacts with oxygen to give off heat is called combustion."},
    {"query": "What is the capital of France?", "ground_truth": "I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook."},
    {"query": "Can you explain linear calculus?", "ground_truth": "I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook."},
    {"query": "Who painted the Mona Lisa?", "ground_truth": "I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook."},
    {"query": "What is the capital of Japan?", "ground_truth": "I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook."},
    {"query": "How does a car engine work?", "ground_truth": "I’m focused on Class 8 Science; try re-phrasing your question or asking something from your textbook."},
    {"query": "What are the properties of synthetic fibres?", "ground_truth": "They dry up quickly, are durable, less expensive, readily available and easy to maintain."},
    {"query": "What are chemical fertilizers?", "ground_truth": "Fertilizers are chemical substances which are rich in a particular nutrient, produced in factories like urea and ammonium sulphate."},
    {"query": "Define ignition temperature.", "ground_truth": "The lowest temperature at which a substance catches fire is called its ignition temperature."},
    {"query": "What are endangered animals?", "ground_truth": "Animals whose numbers are diminishing to a level that they might face extinction are known as endangered animals."},
    {"query": "What is lateral inversion?", "ground_truth": "An effect where the left side of an object appears on the right side and the right side appears on the left side in a mirror."}
]

In [34]:
def run_evaluation_suite(api_key):
    tutor = AITutorRAGPipeline()
    print("📊 Initiating Automated Evaluation System Across 20 Queries...")
    
    with open("evaluation_report.csv", mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Query", "Expected Ground Truth", "System Generated Response", "BLEU Score", "ROUGE-L Score", "Review Status"])
        
        for idx, case in enumerate(eval_dataset, start=1):
            q, gt = case["query"], case["ground_truth"]
            
            output = tutor.generate_response(q, api_key)
            gen = output["answer"]
            
            bleu = sentence_bleu([gt.lower().split()], gen.lower().split(), smoothing_function=smooth_func)
            rouge = scorer.score(gt, gen)['rougeL'].fmeasure
            
            status = "Pass" if rouge > 0.45 or "focused on Class 8" in gen else "Needs Alignment Review"
            writer.writerow([q, gt, gen, round(bleu, 4), round(rouge, 4), status])
            print(f"[{idx}/20] Checked Query: '{q[:25]}...' -> ROUGE-L: {round(rouge, 4)}")
            
    print("\n✨ Metrics tracking finished! Results saved to 'evaluation_report.csv'")

if __name__ == "__main__":
    if 'ipykernel' in sys.modules:
        # ◄── REPLACE THIS with your actual OpenAI API Key string before running!
        MY_OPENAI_KEY = "your-api-key-here" 
        run_evaluation_suite(MY_OPENAI_KEY)
    else:
        if len(sys.argv) > 1:
            run_evaluation_suite(sys.argv[1])
        else:
            print("Usage Error: Please supply an API Key string.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4184.96it/s]


📊 Initiating Automated Evaluation System Across 20 Queries...
[1/20] Checked Query: 'What are the two main cro...' -> ROUGE-L: 0.0
[2/20] Checked Query: 'Define pasteurisation....' -> ROUGE-L: 0.0317
[3/20] Checked Query: 'What is a polymer?...' -> ROUGE-L: 0.0
[4/20] Checked Query: 'Name a liquid metal....' -> ROUGE-L: 0.0
[5/20] Checked Query: 'What is the process of lo...' -> ROUGE-L: 0.0385
[6/20] Checked Query: 'What are pathogens?...' -> ROUGE-L: 0.0
[7/20] Checked Query: 'What is calorific value?...' -> ROUGE-L: 0.0
[8/20] Checked Query: 'What is an ecosystem?...' -> ROUGE-L: 0.0
[9/20] Checked Query: 'What is weeding?...' -> ROUGE-L: 0.0
[10/20] Checked Query: 'Define combustion....' -> ROUGE-L: 0.0357
[11/20] Checked Query: 'What is the capital of Fr...' -> ROUGE-L: 0.0702
[12/20] Checked Query: 'Can you explain linear ca...' -> ROUGE-L: 0.0702
[13/20] Checked Query: 'Who painted the Mona Lisa...' -> ROUGE-L: 0.0702
[14/20] Checked Query: 'What is the capital of Ja...' -> ROUG